# 02. 개선 정의 적용 검증

`01_original_reproduction.ipynb` 에서 확인한 문제를 수정한 분석이다.
**원본 결과를 대체하지 않으며**, 감사를 위한 보조 검증으로만 사용한다.

| # | 수정 항목 | 원본 | 개선 |
|---|---|---|---|
| 1 | 분석 단위 | 행(90,786) | 고유 신고(64,517) 병행 |
| 2 | 제품 역할 | Suspect + Concomitant | Suspect 한정 병행 |
| 3 | 연령 환산 | Day(s)=0, 정수 절삭 | 실제 환산, 절삭 없음 |
| 4 | 연령 구간 | 함수/pd.cut 이원화 | 단일 기준, 라벨과 경계 일치 |
| 5 | 심각도 비율 | 도넛 중복 합산(43.9%) | 행/신고 단위 비중복 |
| 6 | 브랜드 식별 | 제품명 정확 일치 | 문자열 패턴 기반 브랜드군 통합 |
| 7 | 이상치 | 하드코딩 목록 | 심각 비율 분포의 탐색적 IQR 비교 |
| 8 | 외부 검증 | 국내 보도자료 인용 | FDA 공식 문서 대조 |

**이 노트북 실행 시 생성되는 파일**

`results/` — `metric_comparison.csv` · `data_unit_sensitivity.csv` ·
`brand_name_normalization.csv` · `external_validation.csv` · `improved_summary.csv`

`figures/` — `imp_01_industry_unit_compare.png` · `imp_02_brand_iqr_compare.png` ·
`imp_03_brand_yearly.png`


In [ ]:
# ── 원본 데이터 경로 ──────────────────────────────────────────────────
# 아래 한 줄만 본인 환경에 맞게 수정하면 됩니다.
# 또는 셸에서 export ER_CAERS_CSV=/path/to/CAERS_ASCII_2004_2017Q2.csv
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

os.environ.setdefault("ER_CAERS_CSV", "data/CAERS_ASCII_2004_2017Q2.csv")

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib          # 한글 폰트
except ImportError:
    print("[안내] pip install -r requirements.txt 를 실행하세요")
plt.rcParams.update({"figure.dpi":110, "savefig.dpi":130, "figure.facecolor":"white",
                     "savefig.facecolor":"white", "axes.unicode_minus":False})

# ── 저장 위치: 실행 디렉터리와 무관하게 프로젝트 루트를 판별 ──────────
def find_project_root(start=None):
    """requirements.txt 가 있는 디렉터리를 프로젝트 루트로 본다."""
    cur = Path(start or os.getcwd()).resolve()
    for cand in [cur, *cur.parents]:
        if (cand / "requirements.txt").exists() and (cand / "notebooks").is_dir():
            return cand
    return cur.parent if cur.name == "notebooks" else cur

PROJECT_ROOT = find_project_root()
FIG_DIR = PROJECT_ROOT / "figures"
RES_DIR = PROJECT_ROOT / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

# 상대경로면 프로젝트 루트 기준으로 해석
_p = Path(os.environ["ER_CAERS_CSV"])
CSV_PATH = _p if _p.is_absolute() else (PROJECT_ROOT / _p)
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"원본 CSV를 찾을 수 없습니다: {CSV_PATH} | "
        "data/README.md 의 안내에 따라 CAERS 원본을 data/ 에 두거나, "
        "os.environ['ER_CAERS_CSV'] 에 실제 경로를 지정하세요.")

print("CSV       :", CSV_PATH)
print("루트      :", PROJECT_ROOT)
print("저장 위치 :", RES_DIR, "|", FIG_DIR)

In [ ]:
df = pd.read_csv(CSV_PATH)
df.rename(columns={"PRI_Reported Brand/Product Name":"products_name",
                   "SYM_One Row Coded Symptoms":"symptoms",
                   "CI_Gender":"gender",
                   "CI_Age at Adverse Event":"age",
                   "CI_Age Unit":"age_unit",
                   "RA_Report #":"ra_report",
                   "RA_CAERS Created Date":"created_date",
                   "AEC_Event Start Date":"start_date",
                   "PRI_Product Role":"products_role",
                   "PRI_FDA Industry Code":"industry_code",
                   "AEC_One Row Outcomes":"outcomes",
                   "PRI_FDA Industry Name":"products_types"}, inplace=True)

SERIOUS_KEYS = ['DEATH', 'HOSPITALIZATION', 'LIFE THREATENING']   # 원본 정의 유지
df['serious'] = df['outcomes'].apply(
    lambda o: False if pd.isna(o) else any(k in o for k in SERIOUS_KEYS))
df['created_date'] = pd.to_datetime(df['created_date'], format='%m/%d/%Y')
df['year'] = df.created_date.dt.year

print(f"행 {len(df):,} · 고유 신고 {df.ra_report.nunique():,} · "
      f"Suspect {(df.products_role=='Suspect').sum():,}")

## 1. 행과 신고의 관계

한 신고(`ra_report`)가 여러 행으로 나타나는 원인을 확인한다.

In [ ]:
g = df.groupby('ra_report')
sizes = g.size()
print("신고당 행 수 분포")
print(sizes.value_counts().sort_index().head(8).to_string())
print(f"\n1행 신고 {int((sizes==1).sum()):,}건 / 다중행 신고 {int((sizes>1).sum()):,}건")

multi = df[df.ra_report.isin(sizes[sizes>1].index)]
mg = multi.groupby('ra_report')
print("\n다중행 신고 안에서 값이 2종 이상인 비율")
for col, label in [('products_name','제품명'), ('products_types','산업군'),
                   ('products_role','제품 역할'), ('outcomes','부작용 결과')]:
    print(f"  {label:10s} {(mg[col].nunique()>1).mean()*100:5.1f}%")

print("\n→ outcomes 는 신고 단위로 부여되며 제품 행마다 복제된다.")
print("→ 따라서 행 단위 집계는 제품 수가 많은 신고를 그만큼 중복 반영한다.")

## 2. 분석 단위별 심각도 비율

In [ ]:
uniq  = df.groupby('ra_report').serious.any()
sus   = df[df.products_role == 'Suspect']
sus_u = sus.groupby('ra_report').serious.any()

rows = [("행 기준 (원본)", len(df), int(df.serious.sum())),
        ("고유 신고 기준", len(uniq), int(uniq.sum())),
        ("Suspect 행 기준", len(sus), int(sus.serious.sum())),
        ("Suspect 고유 신고", len(sus_u), int(sus_u.sum()))]
print(f"{'기준':20s} {'관측 수':>10} {'심각':>9} {'비율':>8}")
for lbl, n, s in rows:
    print(f"{lbl:20s} {n:>10,} {s:>9,} {s/n*100:7.2f}%")

## 3. 심각도 비율 재계산

보고서의 43.9%가 성립하지 않는 이유를 수치로 확인한다.

In [ ]:
donut = pd.Series({
    '사망': df['outcomes'].str.contains('DEATH', na=False).sum(),
    '생명위협': df['outcomes'].str.contains('LIFE THREATENING', na=False).sum(),
    '입원': df['outcomes'].str.contains('HOSPITALIZATION', na=False).sum(),
    '병원방문(ER+진료)': df['outcomes'].str.contains('VISIT', na=False).sum(),
    '비심각': int((~df.serious).sum())})
donut_total = int(donut.sum())
serious_donut = 100 - 100*donut['비심각']/donut_total

print(f"도넛 분모(슬라이스 합) {donut_total:,}  vs  실제 행 수 {len(df):,}")
print(f"중복 계상 {donut_total-len(df):,}행")
print(f"심각 슬라이스 합 {serious_donut:.1f}%  ← 보고서의 43.9%\n")

print("① 중복 없는 행 단위 심각도")
print(f"   {df.serious.sum():,} / {len(df):,} = {df.serious.mean()*100:.2f}%\n")
print("② 중복 없는 신고 단위 심각도")
print(f"   {uniq.sum():,} / {len(uniq):,} = {uniq.mean()*100:.2f}%\n")
print("③ 개별 outcome 보유율 (행 기준, 항목 간 중복 허용)")
for k, pat in [('사망','DEATH'), ('생명위협','LIFE THREATENING'), ('입원','HOSPITALIZATION'),
               ('ER 방문','VISITED AN ER'), ('진료 방문','VISITED A HEALTH CARE PROVIDER')]:
    n = df['outcomes'].str.contains(pat, na=False).sum()
    print(f"   {k:10s} {n:>7,}  {n/len(df)*100:5.2f}%")
print("\n→ ③의 합은 100%를 넘는다. 한 신고가 여러 결과를 동시에 가질 수 있기 때문이다.")
print("→ '전체 신고 중 XX%가 심각' 형태의 문장에는 ① 또는 ②만 사용할 수 있다.")

## 4. 연령 환산 개선

원본은 `Day(s)` 를 모두 0세로 처리하고 주·월 단위를 정수 절삭했다.
개선안은 실제 비율로 환산하고 절삭하지 않는다.

**구간 경계** — 라벨과 실제 경계를 일치시키기 위해 `right=False` 를 사용한다.
`right=True` 를 쓰면 6세가 영유아, 20세가 청소년, 60세가 성인으로 분류되어
라벨(`영유아(0-5)`, `청소년(6-19)`, `성인(20-59)`)과 어긋난다.

In [ ]:
def convert_age_improved(x):
    age, unit = x['age'], x['age_unit']
    if pd.isna(age) or pd.isna(unit) or unit == 'Not Available':
        return np.nan
    per_year = {'Day(s)':365.25, 'Week(s)':52.18, 'Month(s)':12.0,
                'Year(s)':1.0, 'Decade(s)':0.1}
    if unit not in per_year:
        return np.nan
    y = age / per_year[unit]
    return np.nan if y > 120 else y

def convert_age_original(x):
    age, unit = x['age'], x['age_unit']
    if unit == 'Day(s)': r = 0
    elif unit == 'Week(s)': r = int(age/52)
    elif unit == 'Month(s)': r = int(age/12)
    elif unit == 'Year(s)': r = age
    elif unit == 'Decade(s)': r = age*10
    else: return None
    return None if r > 120 else r

df['age_year_improved'] = df.apply(convert_age_improved, axis=1)
df['age_year_original'] = df.apply(convert_age_original, axis=1)

changed = int((df.age_year_original.notna() &
               (df.age_year_original != df.age_year_improved)).sum())
print(f"연속 환산값이 달라지는 행: {changed:,}")
print(df[df.age_year_original.notna() &
         (df.age_year_original != df.age_year_improved)].groupby('age_unit').size().to_string())

In [ ]:
LAB = ['영유아(0-5)', '청소년(6-19)', '성인(20-59)', '노인(60+)']

# 라벨과 경계를 일치시킨 구간화: [0,6) [6,20) [20,60) [60,120]
age_group_improved = pd.cut(
    df['age_year_improved'],
    bins=[0, 6, 20, 60, 120.000001],
    labels=LAB,
    right=False,
    include_lowest=True)

imp = age_group_improved.value_counts()
orig = df.age_year_original.apply(
    lambda a: None if pd.isna(a) else ('영유아(0-5)' if a <= 5 else '청소년(6-19)' if a <= 19
                                       else '성인(20-59)' if a <= 59 else '노인(60+)')).value_counts()

print(f"{'구간':14s} {'원본':>9} {'개선':>9} {'차이':>8}")
for k in LAB:
    print(f"{k:14s} {orig.get(k,0):>9,} {imp.get(k,0):>9,} {imp.get(k,0)-orig.get(k,0):>+8,}")
print(f"{'합계':14s} {orig.sum():>9,} {imp.sum():>9,}")

print("\n경계 확인")
for v in [0, 5.5, 6, 19.9, 20, 59.9, 60, 120]:
    lab = pd.cut(pd.Series([float(v)]), bins=[0,6,20,60,120.000001],
                 labels=LAB, right=False, include_lowest=True)[0]
    print(f"  {v:>6}세 → {lab}")

print(f"\n→ 연속 환산값은 {changed:,}행에서 달라지지만, 4구간 집계 결과는 원본과 동일하다.")
print("→ 이 데이터에서는 환산 방식 개선이 연령대 분류를 바꾸지 않는다.")

## 5. 산업군 — 행 기준 vs 고유 신고 기준

In [ ]:
tot  = df.groupby('products_types').size()
sev  = df[df.serious].groupby('products_types').size()
rate = (sev/tot*100).fillna(0)
top10 = rate[tot >= 100].sort_values(ascending=False).head(10)

u = df.groupby(['products_types','ra_report']).serious.any().reset_index()
utot = u.groupby('products_types').size()
usev = u[u.serious].groupby('products_types').size()
urate = (usev/utot*100).fillna(0)
utop10 = urate[utot >= 100].sort_values(ascending=False).head(10)

cmp = pd.DataFrame({'행 기준(%)': rate[top10.index].round(2),
                    '고유 신고(%)': urate[top10.index].round(2),
                    '행 n': tot[top10.index], '신고 n': utot[top10.index]})
cmp['차이(%p)'] = (cmp['고유 신고(%)'] - cmp['행 기준(%)']).round(2)
print(cmp.to_string())

print("\n순위 변동")
print(f"  행 기준 상위10에만: {[k[:30] for k in top10.index if k not in utop10.index]}")
print(f"  고유 기준 상위10에만: {[k[:30] for k in utop10.index if k not in top10.index]}")

fig, ax = plt.subplots(figsize=(12.5, 5.5))
x = np.arange(len(top10))
ax.bar(x-0.2, rate[top10.index].values, width=0.4, color='#E24B4A', alpha=.85, label='행 기준')
ax.bar(x+0.2, urate[top10.index].values, width=0.4, color='#185FA5', alpha=.75, label='고유 신고 기준')
ax.set_xticks(x); ax.set_xticklabels([l[:20] for l in top10.index], rotation=40, ha='right')
ax.set_ylabel('심각 부작용 비율 (%)')
ax.set_title('산업군별 심각 비율 — 분석 단위 비교 [개선]', fontweight='bold', fontsize=13)
ax.legend(); ax.yaxis.grid(True, linestyle='--', alpha=.4)
plt.tight_layout()
plt.savefig(FIG_DIR / 'imp_01_industry_unit_compare.png', bbox_inches='tight')
plt.show()

## 6. 문자열 패턴 기반 브랜드군 통합

`HYDROXYCUT` 처럼 하나의 브랜드가 여러 제품명으로 흩어져 있다.

**이 처리는 공인 브랜드 사전을 사용한 정규화가 아니다.** 제품명에 특정 문자열이
포함되는지만 확인하는 규칙 기반 통합이며, 다음 한계를 갖는다.

- 동일 브랜드라도 표기가 크게 다르면 누락된다
- 다른 브랜드가 같은 문자열을 포함하면 과대 포함될 수 있다
- 제조사 단위인지 제품 라인 단위인지 구분하지 않는다

따라서 기존 분석의 표본 누락 규모를 확인하는 **감사 목적**으로만 사용하고,
새로운 주 분석의 집계 단위로 삼지 않는다.

In [ ]:
PATTERNS = {'HYDROXYCUT': (r'HYDROXY\s*CUT', 'HYDROXYCUT'),
            'OXYELITE': (r'OXY\s*ELITE', 'OXYELITE PRO'),
            'HERBALIFE': (r'HERBALIFE', 'HERBALIFE CELL ACTIVATOR'),
            'PLEXUS': (r'PLEXUS', 'PLEXUS SLIM'),
            'ALL DAY ENERGY GREENS': (r'ALL DAY ENERGY GREENS', 'ALL DAY ENERGY GREENS'),
            'RAW OYSTERS': (r'RAW OYSTER', 'RAW OYSTERS')}

def brand_rows(pat):
    return df[df.products_name.str.contains(pat, na=False, regex=True, case=False)]

rows = []
for canon, (pat, used) in PATTERNS.items():
    m = brand_rows(pat); e = df[df.products_name == used]
    rows.append({'브랜드군': canon, '기존 차트 사용 제품명': used,
                 '통합 전 행 수': len(e), '통합 전 심각 수': int(e.serious.sum()),
                 '통합 전 심각비율(%)': round(e.serious.mean()*100, 2) if len(e) else np.nan,
                 '통합 후 행 수': len(m), '통합 후 심각 수': int(m.serious.sum()),
                 '통합 후 심각비율(%)': round(m.serious.mean()*100, 2),
                 '통합 후 고유 신고 수': m.ra_report.nunique(),
                 '통합 후 고유신고 심각비율(%)': round(m.groupby('ra_report').serious.any().mean()*100, 2),
                 '매칭 제품명 수': m.products_name.nunique(),
                 '행 수 배율': round(len(m)/len(e), 2) if len(e) else np.nan,
                 '상위 매칭 예시': " | ".join(m.products_name.value_counts().head(3).index)})
brand_norm = pd.DataFrame(rows)
brand_norm.to_csv(RES_DIR / 'brand_name_normalization.csv', index=False, encoding='utf-8-sig')
print(brand_norm[['브랜드군','통합 전 행 수','통합 전 심각비율(%)',
                  '통합 후 행 수','통합 후 심각비율(%)','매칭 제품명 수']].to_string(index=False))

print("\nHYDROXYCUT 상위 매칭 제품명")
h = brand_rows(r'HYDROXY\s*CUT')
print(h.products_name.value_counts().head(8).to_string())

## 7. 심각 비율 분포의 탐색적 IQR 비교

원본에는 이상치 탐지 알고리즘이 없었다. 여기서는 **"통계적 이상치"라는 표현이
성립하는지 확인하기 위한 보조 검증**으로 IQR과 robust z-score를 계산한다.

**주의 — 이 결과를 위험 브랜드 판정 기준으로 사용하지 않는다.**

- 브랜드별 표본 수가 20행에서 수백 행까지 크게 다르다. 표본이 작을수록 비율의
  분산이 커지므로, 표본 수를 보정하지 않은 비율 간 비교다
- 신고 데이터는 자발신고이며 브랜드별 노출량(판매량)을 반영하지 않는다
- 따라서 이 계산은 **원본 주장을 감사하기 위한 참고 지표**이며,
  새로운 위험 판정 결과가 아니다

In [ ]:
brand = (df.groupby('products_name')
           .agg(total=('serious','size'), serious=('serious','sum'))
           .query('total >= 20').loc[lambda x: x.index != 'REDACTED'])
brand['ratio'] = brand.serious/brand.total*100
print(f"표본 20행 이상 브랜드 {len(brand)}종 (원본은 여기서 상위 80개만 사용)")
print(f"심각비율 분포: min {brand.ratio.min():.1f} / Q1 {brand.ratio.quantile(.25):.1f} / "
      f"중앙 {brand.ratio.median():.1f} / Q3 {brand.ratio.quantile(.75):.1f} / max {brand.ratio.max():.1f}")

q1, q3 = brand.ratio.quantile([.25, .75]); iqr = q3-q1
upper = q3 + 1.5*iqr
med = brand.ratio.median(); mad = (brand.ratio-med).abs().median()
brand['robust_z'] = 0.6745*(brand.ratio-med)/mad if mad else np.nan
brand['IQR_상한초과'] = brand.ratio > upper
brand['robust_z_3.5초과'] = brand.robust_z > 3.5

print(f"\nIQR 상한 {upper:.1f}%  → 초과 {int(brand['IQR_상한초과'].sum())}종")
print(f"robust z > 3.5        → 초과 {int(brand['robust_z_3.5초과'].sum())}종")

원본라벨 = ['HERBALIFE CELL ACTIVATOR','PLEXUS SLIM','OXYELITE PRO',
          'ALL DAY ENERGY GREENS','RAW OYSTERS','HYDROXYCUT']
print("\n원본이 라벨로 강조한 브랜드의 위치")
print(f"  {'브랜드':26s} {'표본':>5} {'비율':>6} {'IQR초과':>7} {'z':>6}")
for b in 원본라벨:
    if b in brand.index:
        r = brand.loc[b]
        print(f"  {b:26s} {int(r.total):>5} {r.ratio:6.1f} "
              f"{'O' if r['IQR_상한초과'] else 'X':>7} {r.robust_z:6.2f}")
    else:
        print(f"  {b:26s}  표본 20행 미만")

V = 'Vit/Min/Prot/Unconv Diet(Human/Animal)'
bv = (df[df.products_types == V].groupby('products_name')
        .agg(total=('serious','size'), serious=('serious','sum'))
        .query('total >= 20').loc[lambda x: x.index != 'REDACTED'])
bv['ratio'] = bv.serious/bv.total*100
q1v, q3v = bv.ratio.quantile([.25,.75]); upv = q3v + 1.5*(q3v-q1v)
print(f"\n[보고서가 명시한 Vit/Min 산업군 한정 {len(bv)}종] IQR 상한 {upv:.1f}%")
for b in 원본라벨:
    if b in bv.index:
        print(f"  {b:26s} {bv.loc[b,'ratio']:6.1f}%  IQR초과 {'O' if bv.loc[b,'ratio']>upv else 'X'}")

print("\n→ 심층 분석 대상 OXYELITE PRO·HYDROXYCUT은 어느 기준으로도 상한을 넘지 않는다.")
print("→ 두 브랜드는 당시 산점도 우상단을 육안으로 판별해 선정한 사례이며,")
print("   통계적 이상치가 아니다. '통계적 이상치 탐지'라는 표현은 성립하지 않는다.")

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 7))
below = brand[~brand['IQR_상한초과']]; above = brand[brand['IQR_상한초과']]
ax.scatter(below.total, below.ratio, s=40, color='#C7CFDB',
           edgecolors='white', linewidths=.5, label='IQR 상한 이하')
ax.scatter(above.total, above.ratio, s=90, color='#F2B441',
           edgecolors='white', linewidths=.8, label=f'IQR 상한 초과 ({len(above)}종)')
ax.axhline(upper, ls='--', color='#F2B441', lw=1.2)
ax.text(brand.total.max(), upper, f'  IQR 상한 {upper:.1f}%', va='center',
        fontsize=11, color='#B8860B')
hl = brand[brand.index.isin(원본라벨)]
ax.scatter(hl.total, hl.ratio, s=150, color='#E24B4A',
           edgecolors='white', linewidths=1.0, zorder=5, label='원본 강조 브랜드')
for b, r in hl.iterrows():
    ax.annotate(b[:22], (r.total, r.ratio), textcoords='offset points',
                xytext=(7, 4), fontsize=9, fontweight='bold')
ax.set_xscale('log')
ax.set_xlabel('총 신고 건수 (행 기준, 로그 스케일)')
ax.set_ylabel('심각 부작용 비율 (%)')
ax.set_title('브랜드별 심각 비율 분포 — 탐색적 IQR 비교 [보조 검증]',
             fontweight='bold', fontsize=13)
ax.legend(); ax.yaxis.grid(True, linestyle='--', alpha=.4)
fig.text(0.01, -0.02, '표본 수가 서로 다른 비율의 탐색적 비교이며, 위험 판정 기준이 아님',
         fontsize=11, color='#5e6672')
plt.tight_layout()
plt.savefig(FIG_DIR / 'imp_02_brand_iqr_compare.png', bbox_inches='tight')
plt.show()

## 8. 외부 검증 — FDA 공식 자료 대조

당시 사례 분석은 국내 보도자료를 인용했다. 여기서는 **FDA 공식 문서**를 근거로
조치 시점과 데이터상 신고 추이를 대조한다.

**시점의 일치는 인과관계가 아니라 외부 사건과의 시간적 대조다.** 자발신고
데이터는 보도 이후 신고가 늘어나는 보고 편향을 갖는다.

In [ ]:
FDA_SOURCES = [
    {'브랜드군': 'HYDROXYCUT',
     '조치 내용': '간독성 문제로 자발적 회수(voluntary recall)',
     '조치 시점': '2009-05',
     'FDA 문서': 'Data Mining at FDA — White Paper',
     'URL': 'https://www.fda.gov/science-research/data-mining/data-mining-fda-white-paper'},
    {'브랜드군': 'OXYELITE',
     '조치 내용': 'FDA 경고 및 회수(recall)',
     '조치 시점': '2013-11-09',
     'FDA 문서': 'Annual Report to Congress on the Use of Mandatory Recall Authority (2014)',
     'URL': 'https://www.fda.gov/food/food-safety-modernization-act-fsma/'
            'annual-report-congress-use-mandatory-recall-authority-2014'},
]

records = []
for src in FDA_SOURCES:
    canon = src['브랜드군']
    pat = PATTERNS[canon][0]
    m = brand_rows(pat)
    y = m.groupby('year').agg(행=('serious','size'), 심각=('serious','sum'),
                              신고=('ra_report','nunique'))
    act_year = int(src['조치 시점'][:4])
    peak_year = int(y.행.idxmax())
    prev = int(y.행.get(act_year-1, 0)); cur = int(y.행.get(act_year, 0))

    print(f"\n=== {canon} ===")
    print(f"FDA 조치: {src['조치 시점']} · {src['조치 내용']}")
    print(y.to_string())
    print(f"조치 전년({act_year-1}) {prev}행 → 조치 연도({act_year}) {cur}행"
          f"{f' ({cur/prev:.1f}배)' if prev else ''}")
    print(f"최대 신고 연도: {peak_year}년 {int(y.행.max())}행 "
          f"{'← 조치 연도와 일치' if peak_year == act_year else '← 조치 연도와 불일치'}")

    for yr, r in y.iterrows():
        records.append({**src, '연도': int(yr), '행 수': int(r.행),
                        '심각 행 수': int(r.심각), '고유 신고 수': int(r.신고),
                        '조치 연도 여부': 'Y' if yr == act_year else '',
                        '최대 신고 연도 여부': 'Y' if yr == peak_year else ''})

ext = pd.DataFrame(records)
ext.to_csv(RES_DIR / 'external_validation.csv', index=False, encoding='utf-8-sig')
print(f"\nexternal_validation.csv 저장 ({len(ext)}행)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ACTION = {'HYDROXYCUT': 2009, 'OXYELITE': 2013}
for ax, canon in zip(axes, ['HYDROXYCUT', 'OXYELITE']):
    pat, used = PATTERNS[canon]
    m = brand_rows(pat); e = df[df.products_name == used]
    a = m.groupby('year').size(); b = e.groupby('year').size()
    yrs = range(2004, 2018)
    ax.plot(yrs, [a.get(y,0) for y in yrs], marker='o', color='#185FA5',
            lw=2, label=f'브랜드군 통합 (n={len(m):,})')
    ax.plot(yrs, [b.get(y,0) for y in yrs], marker='s', color='#E24B4A',
            lw=2, ls='--', label=f'기존 제품명 (n={len(e):,})')
    ax.axvline(ACTION[canon], color='#6b7280', ls=':', lw=1.6)
    ax.text(ACTION[canon], ax.get_ylim()[1]*0.95, f' FDA 조치 {ACTION[canon]}',
            fontsize=10, color='#4b5563', va='top')
    ax.set_title(f'{canon} 연도별 신고 건수', fontweight='bold')
    ax.set_xlabel('연도'); ax.set_ylabel('신고 건수 (행)')
    ax.legend(); ax.yaxis.grid(True, ls='--', alpha=.4)
fig.text(0.01, -0.02, '시점 대조이며 인과관계를 뜻하지 않음 · 자발신고 데이터의 보고 편향 존재',
         fontsize=11, color='#5e6672')
plt.tight_layout()
plt.savefig(FIG_DIR / 'imp_03_brand_yearly.png', bbox_inches='tight')
plt.show()

print("해석")
print(" OXYELITE  — 조치 연도(2013)와 신고 급증 연도가 겹치며, 최대 신고 연도이기도 하다.")
print(" HYDROXYCUT — 조치 연도(2009)에 신고가 크게 늘었으나 최대치는 2011년이다.")
print("             회수 이후에도 높은 신고가 이어졌으므로,")
print("             '조치 시점과 최대 신고 시점이 일치했다'고 쓸 수 없다.")
print("\n두 사례 모두 시간적 대조이며, 신고 증가의 원인을 규명한 것이 아니다.")

## 9. 검증 산출물 생성

앞선 계산 결과를 CSV로 저장한다. **이 노트북만 실행하면 `results/` 의 5개 파일이
모두 다시 생성된다.**

In [ ]:
# ── data_unit_sensitivity.csv ──────────────────────────────────────
rows = [("전체 심각도 비율(%)", round(df.serious.mean()*100,2), round(uniq.mean()*100,2),
         round(sus.serious.mean()*100,2), round(sus_u.mean()*100,2)),
        ("전체 관측 수", len(df), df.ra_report.nunique(), len(sus), sus.ra_report.nunique()),
        ("심각 관측 수", int(df.serious.sum()), int(uniq.sum()),
         int(sus.serious.sum()), int(sus_u.sum()))]
for ind in top10.index:
    s_ind = sus[sus.products_types == ind]
    su = s_ind.groupby('ra_report').serious.any()
    rows.append((f"[산업군] {ind}", round(rate[ind],2), round(urate.get(ind, np.nan),2),
                 round(s_ind.serious.mean()*100,2) if len(s_ind) else np.nan,
                 round(su.mean()*100,2) if len(su) else np.nan))
for canon in ['HYDROXYCUT', 'OXYELITE']:
    m = brand_rows(PATTERNS[canon][0]); mu = m.groupby('ra_report').serious.any()
    ms = m[m.products_role == 'Suspect']; msu = ms.groupby('ra_report').serious.any()
    rows.append((f"[브랜드군] {canon}", round(m.serious.mean()*100,2), round(mu.mean()*100,2),
                 round(ms.serious.mean()*100,2), round(msu.mean()*100,2)))

unit_sens = pd.DataFrame(rows, columns=["지표","행 수 기준","고유 신고 기준",
                                        "Suspect 한정(행)","Suspect 한정(고유 신고)"])
unit_sens.to_csv(RES_DIR / 'data_unit_sensitivity.csv', index=False, encoding='utf-8-sig')
print(f"data_unit_sensitivity.csv  {len(unit_sens)}행")

In [ ]:
# ── metric_comparison.csv ──────────────────────────────────────────
oxy_e = df[df.products_name == 'OXYELITE PRO']; oxy_n = brand_rows(PATTERNS['OXYELITE'][0])
hyd_e = df[df.products_name == 'HYDROXYCUT'];   hyd_n = brand_rows(PATTERNS['HYDROXYCUT'][0])
cut_orig = pd.cut(df['age_year_original'], bins=[0,6,20,60,120],
                  labels=['영유아(0-5)','청소년(6-19)','성인(20-59)','노인(66+)']).value_counts()

M = [
 ("데이터 규모","전체 행 수","90,786", f"{len(df):,}", f"{len(df):,}","일치","-"),
 ("데이터 규모","고유 RA_Report #","(미기재)", f"{df.ra_report.nunique():,}",
  f"{df.ra_report.nunique():,}","해당없음","보고서는 90,786만 '총 데이터 수'로 제시"),
 ("연령","영유아(0-5)","2,964", f"{orig.get('영유아(0-5)',0):,}", f"{imp.get('영유아(0-5)',0):,}",
  "일치","환산 방식 개선에도 4구간 집계는 동일"),
 ("연령","청소년(6-19)","3,007", f"{orig.get('청소년(6-19)',0):,}", f"{imp.get('청소년(6-19)',0):,}","일치","-"),
 ("연령","성인(20-59)","26,045", f"{orig.get('성인(20-59)',0):,}", f"{imp.get('성인(20-59)',0):,}","일치","-"),
 ("연령","노인(60+)","20,890", f"{orig.get('노인(60+)',0):,}", f"{imp.get('노인(60+)',0):,}","일치","-"),
 ("연령","연속 환산값이 달라지는 행","(해당없음)","0", f"{changed:,}","해당없음",
  "Day(s) 실제 환산, 주·월 절삭 제거. 구간 집계에는 영향 없음"),
 ("연령","스택바용 pd.cut 영유아","(그래프에만 사용)", f"{cut_orig.get('영유아(0-5)',0):,}",
  f"{imp.get('영유아(0-5)',0):,}","불일치",
  "원본 cell13은 bins=[0,6,20,60,120] right=True → 0세 1,212행 누락, 경계 1칸 이동"),
 ("심각도","행 단위 serious 건수","(미기재)", f"{df.serious.sum():,}", f"{df.serious.sum():,}","해당없음","-"),
 ("심각도","행 단위 serious 비율","(미기재)", f"{df.serious.mean()*100:.2f}%",
  f"{uniq.mean()*100:.2f}%","해당없음","개선값은 고유 신고 기준"),
 ("심각도","전체 신고 중 심각 비율","43.9%", f"{serious_donut:.1f}%",
  f"{df.serious.mean()*100:.2f}%","불일치",
  f"43.9%는 도넛 분모(중복 합산 {donut_total:,})에서 산출. 병원방문 포함, 팀 자체 정의와 불일치"),
 ("산업군","Dietary Conv 심각비율","45.1%", f"{rate['Dietary Conv Food/Meal Replacements']:.2f}%",
  f"{urate['Dietary Conv Food/Meal Replacements']:.2f}%","일치","개선값은 고유 신고 기준"),
 ("산업군","Vit/Min 심각비율","34.8%", f"{rate[V]:.2f}%", f"{urate[V]:.2f}%","일치","-"),
 ("산업군","Cosmetics 심각비율","14.0%", f"{rate['Cosmetics']:.2f}%",
  f"{urate['Cosmetics']:.2f}%","일치","-"),
 ("브랜드","OXYELITE PRO 행 수","(미기재)", f"{len(oxy_e):,}", f"{len(oxy_n):,}","해당없음",
  "개선값은 문자열 패턴 기반 브랜드군 통합 후"),
 ("브랜드","OXYELITE PRO 심각 수","(미기재)", f"{int(oxy_e.serious.sum()):,}",
  f"{int(oxy_n.serious.sum()):,}","해당없음","-"),
 ("브랜드","OXYELITE PRO 심각비율","(미기재)", f"{oxy_e.serious.mean()*100:.1f}%",
  f"{oxy_n.serious.mean()*100:.1f}%","해당없음","-"),
 ("브랜드","HYDROXYCUT 행 수","(미기재)", f"{len(hyd_e):,}", f"{len(hyd_n):,}","해당없음","통합 시 7.6배"),
 ("브랜드","HYDROXYCUT 심각 수","(미기재)", f"{int(hyd_e.serious.sum()):,}",
  f"{int(hyd_n.serious.sum()):,}","해당없음","-"),
 ("브랜드","HYDROXYCUT 심각비율","(미기재)", f"{hyd_e.serious.mean()*100:.1f}%",
  f"{hyd_n.serious.mean()*100:.1f}%","불일치",
  "브랜드군 통합 시 55.0% → 37.5%. 기존 값은 정확일치 171행만 반영"),
 ("브랜드","이상치 탐지 로직","통계적 이상치 브랜드 발견",
  "없음(중앙값 보조선 + 하드코딩 라벨 6종, 산점도 육안 선별)",
  "탐색적 IQR 비교에서도 두 사례는 상한 이하","불일치",
  "코드에 탐지 알고리즘 부재. 보조 검증에서도 통계적 이상치에 해당하지 않음"),
 ("외부검증","HYDROXYCUT 조치 연도 신고","(국내 보도자료 인용)",
  "2008년 19행 → 2009년 267행", "최대치는 2011년 410행","부분 일치",
  "조치 연도 급증은 확인되나 최대 신고 연도와는 불일치"),
 ("외부검증","OXYELITE 조치 연도 신고","(국내 보도자료 인용)",
  "2012년 12행 → 2013년 100행", "2013년이 최대 신고 연도","일치",
  "FDA 조치 연도와 최대 신고 연도가 겹침"),
]
metric_cmp = pd.DataFrame(M, columns=["구분","항목","PPT/보고서 주장","기존 코드 재현값",
                                      "개선 분석값","일치 여부","차이 원인"])
metric_cmp.to_csv(RES_DIR / 'metric_comparison.csv', index=False, encoding='utf-8-sig')
print(f"metric_comparison.csv  {len(metric_cmp)}행")

In [ ]:
# ── improved_summary.csv ───────────────────────────────────────────
summary = pd.DataFrame([
    ("전체 심각도", f"{df.serious.mean()*100:.2f}% (행)", f"{uniq.mean()*100:.2f}% (신고)"),
    ("보고서 43.9%", "도넛 중복 합산값", "사용 불가"),
    ("연령 4구간", "2,964 / 3,007 / 26,045 / 20,890",
     f"{imp.get(LAB[0],0):,} / {imp.get(LAB[1],0):,} / {imp.get(LAB[2],0):,} / {imp.get(LAB[3],0):,} (동일)"),
    ("Dietary Conv", f"{rate['Dietary Conv Food/Meal Replacements']:.2f}%",
     f"{urate['Dietary Conv Food/Meal Replacements']:.2f}%"),
    ("Vit/Min", f"{rate[V]:.2f}%", f"{urate[V]:.2f}%"),
    ("HYDROXYCUT", f"{hyd_e.serious.mean()*100:.1f}% (171행)",
     f"{hyd_n.serious.mean()*100:.1f}% ({len(hyd_n):,}행, 브랜드군 통합)"),
    ("이상치 탐지", "알고리즘 없음 (육안 선별)",
     f"탐색적 IQR 상한 {upper:.1f}% — 두 사례 모두 이하"),
    ("외부 검증", "국내 보도자료", "FDA 공식 문서 2건 대조"),
], columns=["항목","원본","개선"])
summary.to_csv(RES_DIR / 'improved_summary.csv', index=False, encoding='utf-8-sig')
print(summary.to_string(index=False))

## 10. 생성 파일 확인

In [ ]:
print("results/")
for f in sorted(RES_DIR.glob('*.csv')):
    print(f"  {f.name:36s} {f.stat().st_size:>8,} bytes")
print("\nfigures/")
for f in sorted(FIG_DIR.glob('*.png')):
    print(f"  {f.name:36s} {f.stat().st_size:>8,} bytes")